# 02 — Comprehend an OpenAPI spec

**Goal:** see what 'comprehension' means concretely: a spec goes in, question-shaped tools come out.

We do it twice: first a **hand-rolled miniature** (offline, so you see the mechanics with no magic), then the real thing (live cell).

**Fixture:** `fixtures/petstore-mini.yaml` — 3 operations, one of them credentialed.

In [ ]:
# Live-mode guard: cells that invoke the real gecko CLI run only when you
# opt in AND npx is available. Everything else in this notebook is offline.
#   export GECKO_COOKBOOK_LIVE=1   # to enable live cells
import os
import shutil

GECKO_LIVE = os.environ.get("GECKO_COOKBOOK_LIVE") == "1" and shutil.which("npx") is not None
FIXTURES = None
from pathlib import Path
here = Path.cwd()
while not (here / "pyproject.toml").exists():
    here = here.parent
FIXTURES = here / "cookbook" / "fixtures"
print(f"live mode: {GECKO_LIVE} — fixtures: {FIXTURES}")

## 1. The raw surface — built for humans

In [ ]:
import yaml

spec = yaml.safe_load((FIXTURES / "petstore-mini.yaml").read_text())
print(f"API: {spec['info']['title']} v{spec['info']['version']}")
for path, methods in spec['paths'].items():
    for method, op in methods.items():
        print(f"  {method.upper():6} {path:15} {op['operationId']}")

## 2. A miniature comprehension pass

Turn each operation into a **question-shaped tool**: a name an agent would reach for, the parameters with their constraints, and — critically — the auth requirement **surfaced but never valued** (the key stays outside).

In [ ]:
def comprehend_mini(spec: dict) -> list[dict]:
    tools = []
    for path, methods in spec['paths'].items():
        for method, op in methods.items():
            params = [
                {
                    'name': p['name'],
                    'required': p.get('required', False),
                    'constraints': {k: v for k, v in p.get('schema', {}).items() if k != 'type'},
                }
                for p in op.get('parameters', [])
            ]
            tools.append({
                'tool': op['operationId'],
                'question_shape': op.get('summary', ''),
                'call': f"{method.upper()} {spec['servers'][0]['url']}{path}",
                'params': params,
                'needs_auth': bool(op.get('security')),
            })
    return tools

for tool in comprehend_mini(spec):
    auth = ' [auth required — injected at call time, never shown]' if tool['needs_auth'] else ''
    print(f"\n{tool['tool']}: {tool['question_shape']}{auth}")
    print(f"  {tool['call']}")
    for p in tool['params']:
        req = 'required' if p['required'] else 'optional'
        print(f"    {p['name']:8} {req:8} {p['constraints']}")

## 3. Why the constraints matter

`limit` is capped at 50 **in the spec**. A naive agent learns that at runtime, from a 400 error, after burning a call (or worse, a paid call). A comprehended tool carries the cap into the tool definition — the first call is correct because the surface's rules traveled with it.

Note also what did NOT travel: any credential. `placeOrder` is marked `needs_auth`, and that is ALL the agent ever sees — the key is injected at the transport edge by the application (the same pattern as this repo's provider seam).

## 4. The real thing (live)

In [ ]:
import subprocess

if GECKO_LIVE:
    result = subprocess.run(
        ["npx", "@geckovision/gecko", "add", str(FIXTURES / "petstore-mini.yaml")],
        capture_output=True, text=True, timeout=600,
    )
    print(result.stdout[-4000:] or result.stderr[-4000:])
else:
    print("SKIP (offline): GECKO_COOKBOOK_LIVE=1 to run `gecko add` on the fixture.")

## What you should see

Live: Gecko ingests the spec and reports the operations it comprehended — compare its output with your miniature. The real pass resolves `$ref`s, guards against cycles, generates example calls from schemas (recorded mode), and attaches provenance.

**Next:** `03_recorded_mode_calls` — exercising a surface at $0.